In [1]:
import pandas as pd
import numpy as np

# Just to check everything is working
print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
# Define column names (NSL-KDD has 41 features + label + difficulty score)
columns = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
    "wrong_fragment","urgent","hot","num_failed_logins","logged_in",
    "num_compromised","root_shell","su_attempted","num_root","num_file_creations",
    "num_shells","num_access_files","num_outbound_cmds","is_host_login",
    "is_guest_login","count","srv_count","serror_rate","srv_serror_rate",
    "rerror_rate","srv_rerror_rate","same_srv_rate","diff_srv_rate",
    "srv_diff_host_rate","dst_host_count","dst_host_srv_count",
    "dst_host_same_srv_rate","dst_host_diff_srv_rate","dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate",
    "dst_host_rerror_rate","dst_host_srv_rerror_rate","label","difficulty"
]

# UPDATING THIS PATH to match where my files are on my SSD
train_path = r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\NSL-KDD\KDDTrain+.txt"
test_path  = r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\NSL-KDD\KDDTest+.txt"

train_df = pd.read_csv(train_path, names=columns)
test_df  = pd.read_csv(test_path, names=columns)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (125973, 43)
Test shape: (22544, 43)


In [3]:
import os

folder = r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\NSL-KDD\nsl-kdd"
print(os.listdir(folder))

['index.html', 'KDDTest+.arff', 'KDDTest+.txt', 'KDDTest-21.arff', 'KDDTest-21.txt', 'KDDTest1.jpg', 'KDDTrain+.arff', 'KDDTrain+.txt', 'KDDTrain+_20Percent.arff', 'KDDTrain+_20Percent.txt', 'KDDTrain1.jpg']


In [4]:
train_df.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


In [5]:
# See what attack types/labels exist in the data
print(train_df['label'].value_counts())

label
normal             67343
neptune            41214
satan               3633
ipsweep             3599
portsweep           2931
smurf               2646
nmap                1493
back                 956
teardrop             892
warezclient          890
pod                  201
guess_passwd          53
buffer_overflow       30
warezmaster           20
land                  18
imap                  11
rootkit               10
loadmodule             9
ftp_write              8
multihop               7
phf                    4
perl                   3
spy                    2
Name: count, dtype: int64


In [6]:
# Create a binary label: normal vs attack
train_df['binary_label'] = train_df['label'].apply(lambda x: 'normal' if x == 'normal' else 'attack')
test_df['binary_label']  = test_df['label'].apply(lambda x: 'normal' if x == 'normal' else 'attack')

print(train_df['binary_label'].value_counts())
print(test_df['binary_label'].value_counts())

binary_label
normal    67343
attack    58630
Name: count, dtype: int64
binary_label
attack    12833
normal     9711
Name: count, dtype: int64


In [7]:
# Check for missing values
print("Missing values in train set:")
print(train_df.isnull().sum().sum())

print("\nMissing values in test set:")
print(test_df.isnull().sum().sum())

# Check for duplicate rows
print("\nDuplicate rows in train set:", train_df.duplicated().sum())
print("Duplicate rows in test set:", test_df.duplicated().sum())

Missing values in train set:
0

Missing values in test set:
0

Duplicate rows in train set: 0
Duplicate rows in test set: 0


In [8]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = ['protocol_type', 'service', 'flag']

encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col])
    encoders[col] = le

print("Categorical columns encoded successfully!")
train_df[categorical_cols].head()

Categorical columns encoded successfully!


,protocol_type,service,flag
0,1,20,9
1,2,44,9
2,1,49,5
3,1,24,9
4,1,24,9


In [9]:
def safe_transform(le, series):
    """Encode a column, mapping any unseen category to a new 'unknown' code."""
    classes = list(le.classes_)
    unseen = set(series) - set(classes)
    
    if unseen:
        print(f"Unseen categories found: {unseen}")
        # Extend the encoder's known classes to include 'unseen' as a category
        le_classes = np.append(le.classes_, 'unseen')
        le.classes_ = le_classes
        series = series.apply(lambda x: x if x in classes else 'unseen')
    
    return le.transform(series)

for col in categorical_cols:
    test_df[col] = safe_transform(encoders[col], test_df[col])

print("Test set categorical columns encoded successfully!")
test_df[categorical_cols].head()

Test set categorical columns encoded successfully!


,protocol_type,service,flag
0,1,49,1
1,1,49,1
2,1,20,9
3,0,14,9
4,1,60,2


In [10]:
# Build a clean mapping dictionary from each encoder's known categories
category_maps = {}
for col in categorical_cols:
    classes = list(encoders[col].classes_)
    category_maps[col] = {cat: idx for idx, cat in enumerate(classes)}

# Apply the mapping to the test set, assigning a new code to anything unseen
for col in categorical_cols:
    mapping = category_maps[col]
    unseen_values = test_df.loc[~test_df[col].isin(mapping.keys()), col].unique()
    if len(unseen_values) > 0:
        print(f"{col}: unseen categories found -> {list(unseen_values)}")
    unseen_code = len(mapping)
    test_df[col] = test_df[col].apply(lambda x: mapping.get(x, unseen_code))

print("Test set categorical columns encoded successfully!")
test_df[categorical_cols].head()

protocol_type: unseen categories found -> [np.int64(1), np.int64(0), np.int64(2)]
service: unseen categories found -> [np.int64(49), np.int64(20), np.int64(14), np.int64(60), np.int64(24), np.int64(54), np.int64(19), np.int64(32), np.int64(47), np.int64(6), np.int64(10), np.int64(15), np.int64(28), np.int64(12), np.int64(35), np.int64(59), np.int64(29), np.int64(44), np.int64(7), np.int64(18), np.int64(66), np.int64(69), np.int64(38), np.int64(33), np.int64(2), np.int64(57), np.int64(4), np.int64(37), np.int64(67), np.int64(68), np.int64(11), np.int64(36), np.int64(46), np.int64(26), np.int64(65), np.int64(34), np.int64(21), np.int64(17), np.int64(63), np.int64(51), np.int64(56), np.int64(31), np.int64(55), np.int64(53), np.int64(23), np.int64(13), np.int64(9), np.int64(45), np.int64(0), np.int64(40), np.int64(8), np.int64(42), np.int64(39), np.int64(62), np.int64(58), np.int64(5), np.int64(41), np.int64(52), np.int64(48), np.int64(16), np.int64(1), np.int64(43), np.int64(30), np.int64

,protocol_type,service,flag
0,3,70,11
1,3,70,11
2,3,70,11
3,3,70,11
4,3,70,11


In [11]:
from sklearn.preprocessing import StandardScaler

# Select numeric feature columns (exclude label, binary_label, difficulty)
feature_cols = [col for col in train_df.columns if col not in ['label', 'binary_label', 'difficulty']]

scaler = StandardScaler()
train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
test_df[feature_cols] = scaler.transform(test_df[feature_cols])

print("Features scaled successfully!")
train_df[feature_cols].head()

Features scaled successfully!


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate
0,-0.110249,-0.124706,-0.686785,0.751111,-0.007679,-0.004919,-0.014089,-0.089486,-0.007736,-0.095076,...,-0.324063,-0.818890,-0.782367,-0.280282,0.069972,-0.289103,-0.639532,-0.624871,-0.224532,-0.376387
1,-0.110249,2.219312,0.781428,0.751111,-0.007737,-0.004919,-0.014089,-0.089486,-0.007736,-0.095076,...,0.734343,-1.035688,-1.161030,2.736852,2.367737,-0.289103,-0.639532,-0.624871,-0.387635,-0.376387
2,-0.110249,-0.124706,1.087305,-0.736235,-0.007762,-0.004919,-0.014089,-0.089486,-0.007736,-0.095076,...,0.734343,-0.809857,-0.938287,-0.174417,-0.480197,-0.289103,1.608759,1.618955,-0.387635,-0.376387
3,-0.110249,-0.124706,-0.442083,0.751111,-0.007723,-0.002891,-0.014089,-0.089486,-0.007736,-0.095076,...,-1.533670,1.258754,1.066401,-0.439078,-0.383108,0.066252,-0.572083,-0.602433,-0.387635,-0.345084
4,-0.110249,-0.124706,-0.442083,0.751111,-0.007728,-0.004814,-0.014089,-0.089486,-0.007736,-0.095076,...,0.734343,1.258754,1.066401,-0.439078,-0.480197,-0.289103,-0.639532,-0.624871,-0.387635,-0.376387


In [12]:
import os

# Create a folder for cleaned/processed data (keeps it separate from raw files)
processed_folder = r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\NSL-KDD\processed"
os.makedirs(processed_folder, exist_ok=True)

train_save_path = os.path.join(processed_folder, "NSLKDD_train_cleaned.csv")
test_save_path  = os.path.join(processed_folder, "NSLKDD_test_cleaned.csv")

train_df.to_csv(train_save_path, index=False)
test_df.to_csv(test_save_path, index=False)

print("Cleaned data saved successfully!")
print("Train saved to:", train_save_path)
print("Test saved to:", test_save_path)

Cleaned data saved successfully!
Train saved to: C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\NSL-KDD\processed\NSLKDD_train_cleaned.csv
Test saved to: C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\NSL-KDD\processed\NSLKDD_test_cleaned.csv


In [13]:
import pandas as pd
df = pd.read_csv(r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\NSL-KDD\processed\NSLKDD_train_cleaned.csv")
print(df.columns.tolist())

['duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'label', 'difficulty', 'binary_label']


In [14]:
print(df['label'].unique())
print(df['binary_label'].unique())

['normal' 'neptune' 'warezclient' 'ipsweep' 'portsweep' 'teardrop' 'nmap'
 'satan' 'smurf' 'pod' 'back' 'guess_passwd' 'ftp_write' 'multihop'
 'rootkit' 'buffer_overflow' 'imap' 'warezmaster' 'phf' 'land'
 'loadmodule' 'spy' 'perl']
['normal' 'attack']


In [15]:
print(r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\NSL-KDD\KDDTrain+.txt")


C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\NSL-KDD\KDDTrain+.txt
